# OfflineMedia Portable Video AI

GitHub-triggered Colab GPU worker for `CAption11/offlinemedia`. This notebook pulls the active branch, consumes a queued request, starts ComfyUI, generates a video, validates the output, and displays it in Colab.

In [ ]:
import os, subprocess, sys, pathlib
REPO='https://github.com/CAption11/offlinemedia.git'
BRANCH='claude/scan-repo-chatgpt-review-6nljgh'
ROOT=pathlib.Path('/content/offlinemedia')
if not ROOT.exists():
    subprocess.run(['git','clone','-b',BRANCH,REPO,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(ROOT),'reset','--hard',f'origin/{BRANCH}'],check=True)
os.chdir(ROOT)
print('Repository:',ROOT)
print('Branch:',BRANCH)

In [ ]:
import shutil, subprocess, sys
print('Python:',sys.version)
if not shutil.which('nvidia-smi'):
    raise RuntimeError('No NVIDIA GPU runtime. In Colab choose Runtime > Change runtime type > GPU.')
subprocess.run(['nvidia-smi'],check=False)

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt','huggingface_hub'],check=True)
print('Dependencies installed.')

In [ ]:
GITHUB_TOKEN=os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')
try:
    from google.colab import userdata
    if not GITHUB_TOKEN:
        GITHUB_TOKEN=userdata.get('GITHUB_TOKEN')
except Exception:
    pass
print('GitHub token configured:',bool(GITHUB_TOKEN))
if not GITHUB_TOKEN:
    print('WARNING: without GITHUB_TOKEN the queued job cannot be claimed or marked complete safely.')

In [ ]:
from portable.trigger_listener import read_trigger_or_none, wait_for_trigger
trigger, trigger_sha=read_trigger_or_none(token=GITHUB_TOKEN)
if trigger is None:
    print('Queue is empty. Seed a request or run the GitHub Actions trigger.')
else:
    print('Queued trigger:',trigger)
print('Waiting for a queued request...')
trigger=wait_for_trigger(token=GITHUB_TOKEN,poll_seconds=15,claim=bool(GITHUB_TOKEN))
print('Received trigger:',trigger)

In [ ]:
from portable.comfyui_bootstrap import bootstrap
comfy_process=bootstrap()
print('ComfyUI is running.')

In [ ]:
import json, subprocess, sys
from portable.trigger_listener import complete_trigger
mode=trigger.get('mode','text_to_video')
prompt=trigger.get('prompt','A small red ball rolling across a wooden table, natural lighting')
width=int(trigger.get('width',320)); height=int(trigger.get('height',240))
frames=int(trigger.get('frames',17)); fps=int(trigger.get('fps',8))
cmd=[sys.executable,'scripts/test_generation.py','--type',mode,'--prompt',prompt,'--width',str(width),'--height',str(height),'--frames',str(frames),'--fps',str(fps),'--workflow-dir','workflows/official']
print('Running:', ' '.join(cmd))
result=subprocess.run(cmd,text=True,capture_output=True)
print(result.stdout)
if result.stderr: print('STDERR:',result.stderr)
if result.returncode!=0:
    if GITHUB_TOKEN:
        complete_trigger(trigger['job_id'],status='failed',token=GITHUB_TOKEN,
                         detail=f'generation exit code {result.returncode}')
        print('Trigger marked failed.')
    raise RuntimeError(f'Generation failed with exit code {result.returncode}')

In [ ]:
from pathlib import Path
from IPython.display import Video, display
from scripts.test_generation import validate_output
from portable.trigger_listener import complete_trigger

paths=[Path(line.split('Output: ',1)[1].strip())
       for line in result.stdout.splitlines() if line.startswith('Output: ')]
if not paths:
    raise RuntimeError('Generation returned no output path.')

# validate_output is the same check the smoke test runs, so the notebook and
# the CLI cannot drift apart. It rejects a missing, empty, non-video or
# single-still-frame output.
try:
    for path in paths:
        print('Validation:', validate_output(path))
        print('Displaying:', path)
        display(Video(str(path), embed=True))
except Exception as exc:
    if GITHUB_TOKEN:
        complete_trigger(trigger['job_id'],status='failed',token=GITHUB_TOKEN,detail=str(exc))
        print('Trigger marked failed.')
    raise

if GITHUB_TOKEN:
    complete_trigger(trigger['job_id'],status='completed',token=GITHUB_TOKEN,
                     detail=f'{len(paths)} output(s) validated')
    print('Trigger marked completed.')
else:
    print('No token: job left as-is in the queue. Set GITHUB_TOKEN to close it out.')
print('REAL VIDEO GENERATION AND OUTPUT VALIDATION PASSED.')